# Capstone — Which pages should you refresh first?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dawngend/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane:** Freestyle — AI Referral and Generative Engine Optimization (GEO) Opportunity Scoring
**Deployed paper:** `docs/index.html` → served at the URL in `../submission/paper_url.txt`
**Written report:** `work/capstone_report.md` (the 10-section rubric version)

This notebook mirrors the deployed paper. Every number it prints is read from a **committed JSON
receipt**, not recomputed here — so if the notebook and the paper ever disagree, one of them is
reading a stale file and the assertion at the end will say so.

The one thing this notebook does that the paper cannot: it **checks the paper against its own
sources**. The final section re-reads `docs/index.html` and asserts that every headline figure in
the prose matches the receipt it came from.

## 1. Question

*The research question and the decision it supports.*

> **Which pages should a content team review first, when there are more pages than review hours?**

- **Unit of analysis:** one pseudonymized content item (`content_id`) over a trailing 90-day window.
- **Output:** a ranked queue with a score, a confidence band, and reason codes.
- **Who acts:** a content strategist or SEO reviewer with edit rights.
- **Cost of a wrong call:** a false positive burns three to five hours of editorial time on a page
  that was fine. A false negative lets a page with existing visibility keep decaying until recovery
  is expensive. The costs are asymmetric, which is why the system orders review and never triggers
  an edit.

### The scope change, stated first

Week 1 declared the lane as **AI Referral and GEO Opportunity Scoring**: finding pages with strong
search demand but disproportionately low AI referral traffic. The modelled deliverable is narrower,
and the reason is a finding rather than an excuse.

The AI visibility gap is real and measurable in this corpus. What does not exist is a **label** for
it — nothing in the release records whether an AI engine later cited a page. So the gap can be
described and ranked by rule, but it cannot be learned, validated, or leakage-checked. Building a
supervised model on it would have meant inventing a target.

The decay half of the same lane does have a label, so that is the half that carries the modelling.
The cell below quantifies the gap that was left descriptive, so the scope change is visible in
numbers rather than only in prose.

In [1]:
# Section 1 - the AI visibility gap that stayed descriptive, and the label that did not exist.
import json
import re
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents, Path("/content/FlyRank-Machine-Learning-Internship")]:
        if (base / "work" / "outputs" / "data_contract.json").exists():
            return base
    raise FileNotFoundError(
        "Could not locate the repo root. In Colab, clone the repo first:\n"
        "  !git clone https://github.com/Dawngend/FlyRank-Machine-Learning-Internship.git"
    )


ROOT = find_repo_root()
OUT = ROOT / "work" / "outputs"
sys.path.insert(0, str(ROOT / "work" / "scripts"))

RECEIPTS = {
    "contract": OUT / "data_contract.json",
    "baseline": OUT / "baseline_action_score_metrics.json",
    "model": OUT / "model_comparison.json",
    "audit": OUT / "validation_audit.json",
    "playbook": OUT / "action_playbook_metrics.json",
}
missing = [name for name, path in RECEIPTS.items() if not path.exists()]
assert not missing, f"missing receipts: {missing} - re-run the scripts they come from"
R = {name: json.loads(path.read_text(encoding="utf-8")) for name, path in RECEIPTS.items()}
print("receipts loaded:", ", ".join(f"{k} ({p.name})" for k, p in RECEIPTS.items()))

frame = pd.read_csv(ROOT / "data" / "processed" / "refresh_feature_vector.csv")

has_ai = frame["ai_sessions_90d"] > 0
real_demand = frame["impressions_90d"] >= 500
gap = real_demand & ~has_ai

print(f"\n--- the half that stayed descriptive: the AI visibility gap ---")
print(f"pages with any AI referral traffic : {has_ai.sum():,} ({has_ai.mean():.2%})")
print(f"pages with real demand (>=500 imp) : {real_demand.sum():,} ({real_demand.mean():.2%})")
print(f"  ... of those, with zero AI traffic: {gap.sum():,} ({gap.sum() / real_demand.sum():.2%})")
print("-> a large, well-defined population. What is missing is not the population, it is the label:")
print("   nothing in the release records whether an AI engine ever cited a page.")

print(f"\n--- the half that carried the modelling: content decay ---")
print(f"pages carrying a downward trend    : {frame['is_declining_label'].sum():,} "
      f"({frame['is_declining_label'].mean():.2%})")
print("-> a labelled target, so it can be learned, validated, and audited for leakage.")

receipts loaded: contract (data_contract.json), baseline (baseline_action_score_metrics.json), model (model_comparison.json), audit (validation_audit.json), playbook (action_playbook_metrics.json)

--- the half that stayed descriptive: the AI visibility gap ---
pages with any AI referral traffic : 1,930 (6.43%)
pages with real demand (>=500 imp) : 16,726 (55.75%)
  ... of those, with zero AI traffic: 15,033 (89.88%)
-> a large, well-defined population. What is missing is not the population, it is the label:
   nothing in the release records whether an AI engine ever cited a page.

--- the half that carried the modelling: content decay ---
pages carrying a downward trend    : 16,262 (54.21%)
-> a labelled target, so it can be learned, validated, and audited for leakage.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source.** `data/raw/content_refresh_anonymized.csv`, the FlyRank ML Internship starter release:
30,000 pseudonymized content items, 44 columns, 32 clients, trailing 90-day performance window.
Warehouse figures quoted in Week 1 come from the gated `FlyRank/internship-warehouse` release,
partition `month=2026-03`.

**The label.** A page is declining when impressions over the last 30 days fell more than **20%**
against the previous 30 days.

> **Documentation discrepancy.** The published FlyRank paper defines this cut at 10% (p. 5). The
> shipped data cuts at 20%. Reconstructing the label at 10% agrees on 93.3% of rows; at 20%, on
> 100.0%. Flagged rather than silently corrected, because anything built on the documented figure
> will not reproduce.

**Excluded, and why.** Four of the six exclusions were planned in the ML-04 contract. Two were found
by the Week 6 audit and are the more interesting half — see section 3.

In [2]:
# Section 2 - the corpus, the exclusions, and the frame's own boundaries.
audit, playbook, contract = R["audit"], R["playbook"], R["contract"]

print("--- corpus ---")
print(f"rows        : {audit['rows']:,}")
print(f"clients     : {audit['n_clients']}")
print(f"base rate   : {audit['base_rate']:.4f}")
print(f"contract    : v{contract['version']}  lane: {contract['lane']}")
print(f"windows     : feature {contract['windows']['feature_window']}, "
      f"label {contract['windows']['label_window']}")
print(f"sealed test : {contract['windows']['sealed_test_month']} (design recorded, NOT executed)")

print("\n--- excluded fields ---")
exclusions = [
    ("trend_direction", "contract", "the label is derived from it"),
    ("trend_pct", "contract", "the label is derived from it"),
    ("is_declining_label", "contract", "the label itself"),
    ("provider_used", "contract", "out of scope for a refresh decision"),
    ("model_used", "contract", "out of scope for a refresh decision"),
    ("impressions_last_30d", "ML-09 audit", "with the next row, reconstructs the label exactly"),
    ("impressions_prev_30d", "ML-09 audit", "with the row above, reconstructs the label exactly"),
    ("client_id", "design", "grouping only, never a feature"),
]
print(pd.DataFrame(exclusions, columns=["field", "found_by", "why"]).to_string(index=False))

print("\n--- the frame's own boundaries ---")
active = (frame["impressions_90d"] > 0) & (frame["sessions_90d"] > 0)
print(f"rows passing the active-content filter: {active.sum():,} / {len(frame):,} ({active.mean():.1%})")
print("  -> the corpus is survivors only; a page that already died is not in it")
print(f"rows with no prior 30-day window      : {playbook['corpus']['held_out_no_prior_window']:,}")
print(f"  -> of which declining               : {playbook['corpus']['held_out_declining_count']} "
      "(zero, by construction - the label needs a prior window)")

--- corpus ---
rows        : 30,000
clients     : 32
base rate   : 0.5421
contract    : v1.1  lane: Freestyle - AI Referral and GEO Opportunity Scoring
windows     : feature ['2026-01-01', '2026-03-31'], label ['2026-04-01', '2026-04-30']
sealed test : 2026-06 (design recorded, NOT executed)

--- excluded fields ---
               field    found_by                                                why
     trend_direction    contract                       the label is derived from it
           trend_pct    contract                       the label is derived from it
  is_declining_label    contract                                   the label itself
       provider_used    contract                out of scope for a refresh decision
          model_used    contract                out of scope for a refresh decision
impressions_last_30d ML-09 audit  with the next row, reconstructs the label exactly
impressions_prev_30d ML-09 audit with the row above, reconstructs the label exactly
          

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**The question shape.** "Which first" is a ranking question, so every scorer is judged on its
predicted probability at **precision@K**. Accuracy appears nowhere in this project: a queue is never
consumed by thresholding at 0.5, it is consumed top down until the reviewer runs out of time.

**The baseline came first.** A transparent hand rule scoring each page 0 to 100 from five stated
conditions, every point traceable to one sentence. It is scored on the identical test rows of each
fold, so the comparison is a comparison rather than a model measured against a whole-corpus figure.

**The split.** `GroupKFold(n_splits=5)` on `client_id`. This is a correctness issue, not a
preference: 32 clients, one holding 12.3% of the corpus, per-client declining rate from 0.000 to
0.937 against a corpus rate of 0.542. Under a row shuffle the same client sits on both sides, and a
model can score well by inferring the client and predicting that client's base rate.

**A time-aware split is not available.** The starter release is one undated snapshot. The contract's
sealed test month stays a recorded design rather than a claim.

### The leakage audit, and what it found

Week 3 excluded the label-derived columns **by name**. Correct, and insufficient. Week 6 ran three
checks instead of one, and only the third found anything:

1. every numeric column scored alone against the label
2. a shuffled-label null under the grouped design
3. **rebuild the label from first principles and see what shipped that can do it**

Check 3 found that `impressions_last_30d` and `impressions_prev_30d` reconstruct the label on
**30,000 of 30,000 rows**. Neither is damning alone. Their ratio is perfect.

In [3]:
# Section 3 - the audit's findings, read from the receipt.
recon = audit["label_reconstruction"]

print("--- check 1: single-feature scan ---")
flagged = audit["single_feature_auc"]["flagged_as_leak"]
print(f"columns flagged at AUC >= 0.90: {len(flagged)} "
      f"({', '.join(f['feature'] for f in flagged)}) - the label's own source column")
print("every feature actually in the model sits between 0.41 and 0.59. The scan says clean.")

print("\n--- check 2: shuffled-label null ---")
null = audit["shuffled_label_null"]["metrics"]["random_forest"]["roc_auc"]
print(f"random forest ROC-AUC on a permuted label: {null['mean']:.4f} +/- {null['std']:.4f}")
print("-> sits on chance. The harness measures the label, not the folds.")

print("\n--- check 3: rebuild the label from its definition ---")
print(f"documented rule : {recon['documented_rule']}")
print(f"recovered rule  : {recon['recovered_rule']}")
print(f"agreement at -10%: {recon['agreement_by_threshold']['-10%']:.4f}")
print(f"agreement at -20%: {recon['agreement_by_threshold']['-20%']:.4f}")
print(f"exact rows      : {recon['rows_reconstructed_exactly']:,} / {recon['rows_total']:,}")

print("\nwhy check 1 could not have found it:")
for col, auc in recon["roc_auc_of_each_column_alone"].items():
    print(f"  {col:<24} alone : AUC {auc:.4f}")
print(f"  {'their ratio':<24}       : AUC {recon['roc_auc_of_impression_ratio_alone']:.4f}")
print("\n-> a one-feature-at-a-time scan cannot see a leak that lives in an interaction.")

from train_refresh_model import CATEGORICAL, NUMERIC  # noqa: E402
leaking = {"impressions_last_30d", "impressions_prev_30d", "trend_direction", "trend_pct",
           "is_declining_label"}
found = sorted(set(NUMERIC + CATEGORICAL) & leaking)
assert not found, f"LEAK: {found} present in the model feature set"
print(f"[OK] none of the {len(leaking)} label-reconstructing columns is among the "
      f"{len(NUMERIC + CATEGORICAL)} model features.")

--- check 1: single-feature scan ---
columns flagged at AUC >= 0.90: 1 (trend_pct) - the label's own source column
every feature actually in the model sits between 0.41 and 0.59. The scan says clean.

--- check 2: shuffled-label null ---
random forest ROC-AUC on a permuted label: 0.5016 +/- 0.0047
-> sits on chance. The harness measures the label, not the folds.

--- check 3: rebuild the label from its definition ---
documented rule : paper p.5: down when impressions fall >10% vs prev 30d
recovered rule  : down when (imp_last_30d - imp_prev_30d) / imp_prev_30d < -20%
agreement at -10%: 0.9334
agreement at -20%: 1.0000
exact rows      : 30,000 / 30,000

why check 1 could not have found it:
  impressions_last_30d     alone : AUC 0.4857
  impressions_prev_30d     alone : AUC 0.6214
  their ratio                    : AUC 1.0000

-> a one-feature-at-a-time scan cannot see a leak that lives in an interaction.


[OK] none of the 5 label-reconstructing columns is among the 26 model features.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Every scorer below is evaluated on the identical grouped folds, with the base rate in the table so
no precision figure floats free of it.

In [4]:
# Section 4 - the results table and what the split was worth.
grouped = audit["split_comparison"]["grouped_by_client"]["metrics"]
naive = audit["split_comparison"]["naive_random"]["metrics"]
base = audit["base_rate"]

rows = [{"scorer": "base rate", "roc_auc": 0.5,
         "precision@50": round(base, 3), "precision@1000": round(base, 3), "sd@1000": None}]
for name in ["baseline_rule", "logistic_regression", "random_forest", "decision_tree_depth2"]:
    block = grouped[name]
    rows.append({
        "scorer": name,
        "roc_auc": round(block["roc_auc"]["mean"], 3),
        "precision@50": round(block["p@50"]["mean"], 3),
        "precision@1000": round(block["p@1000"]["mean"], 3),
        "sd@1000": round(block["p@1000"]["std"], 3),
    })
print("--- same data, same metric, same grouped split ---")
print(pd.DataFrame(rows).to_string(index=False))

print("\n--- what the client grouping was worth (precision@50) ---")
infl = []
for name, block in audit["split_comparison"]["inflation"].items():
    infl.append({
        "scorer": name,
        "grouped": block["p@50"]["grouped"],
        "naive_shuffle": block["p@50"]["naive_random"],
        "inflation_%": block["p@50"]["relative_inflation_pct"],
    })
order = ["baseline_rule", "decision_tree_depth2", "logistic_regression", "random_forest"]
infl = sorted(infl, key=lambda r: order.index(r["scorer"]))
print(pd.DataFrame(infl).to_string(index=False))
print("\n-> inflation is monotone in model capacity. The hand rule is never fitted and cannot")
print("   benefit at all; each step up in capacity buys more. That is the signature of client")
print("   identity leaking through the split, not of a better model.")

# The two fold-count claims the paper makes, asserted here.
rf50, rule50 = grouped["random_forest"]["p@50"], grouped["baseline_rule"]["p@50"]
rf1k, rule1k = grouped["random_forest"]["p@1000"], grouped["baseline_rule"]["p@1000"]
deep = [a > b for a, b in zip(rf1k["per_fold"], rule1k["per_fold"])]
top = [a > b for a, b in zip(rf50["per_fold"], rule50["per_fold"])]
print(f"\nforest ahead of the rule at p@1000 : {sum(deep)}/{len(deep)} folds")
print(f"forest ahead of the rule at p@50   : {sum(top)}/{len(top)} folds  "
      f"(fold {top.index(False) + 1} goes to the rule, "
      f"{rule50['per_fold'][top.index(False)]:.2f} to {rf50['per_fold'][top.index(False)]:.2f})")
assert all(deep) and sum(top) == 4, "the paper's fold-count claims must hold"
print("[OK] both fold-count claims in the paper hold against the receipt.")

--- same data, same metric, same grouped split ---
              scorer  roc_auc  precision@50  precision@1000  sd@1000
           base rate    0.500         0.542           0.542      NaN
       baseline_rule    0.539         0.616           0.558    0.067
 logistic_regression    0.660         0.788           0.710    0.084
       random_forest    0.671         0.776           0.726    0.054
decision_tree_depth2    0.608         0.628           0.613    0.074

--- what the client grouping was worth (precision@50) ---
              scorer  grouped  naive_shuffle  inflation_%
       baseline_rule    0.616          0.556        -9.74
decision_tree_depth2    0.628          0.640         1.91
 logistic_regression    0.788          0.896        13.71
       random_forest    0.776          0.952        22.68

-> inflation is monotone in model capacity. The hand rule is never fitted and cannot
   benefit at all; each step up in capacity buys more. That is the signature of client
   identity l

## 5. Limitations

*What this work cannot claim.*

Written before a reader has to find them.

| Limitation | The evidence for it |
|---|---|
| **Does not predict that refreshing recovers traffic** | No refresh outcome column exists anywhere in the release. The model ranks by whether impressions already fell. |
| **Corpus is survivors only** | 100% of rows pass the active-content filter. A page that already died is not in the frame and cannot be ranked. |
| **3,388 pages are structurally unrankable** | No prior 30-day window, so they cannot be declining under the label definition. All 3,388 are labelled 0 by construction. |
| **Markedly client-dependent** | Fold metrics swing by up to 0.20 depending on which clients land in the test fold. |
| **Top-of-queue advantage is directional** | Forest leads the rule at p@1000 on 5/5 folds but at p@50 on only 4/5. |
| **Within-queue ordering is not explainable** | The model-derived reason codes fire on the whole queue. They explain the population, not why row 12 sits above row 400. |
| **No sealed evaluation** | The contract defines one; the starter release is undated, so the design is recorded and not executed. |

The cell below asserts the three that are checkable in data, so they cannot quietly stop being true.

In [5]:
# Section 5 - the limitations that are checkable, checked.
outcome_like = {c for c in frame.columns if "after" in c.lower() or "post_refresh" in c.lower()}
assert not outcome_like, f"an outcome column exists after all: {outcome_like}"
print(f"[OK] no post-refresh outcome column exists -> no recovery claim is possible. "
      f"(scanned {len(frame.columns)} columns)")

assert active.all(), "the corpus is meant to be active-content only"
print(f"[OK] {active.mean():.0%} of rows are active content -> the model has never seen a dead page.")

no_window = frame["impressions_prev_30d"] == 0
assert frame.loc[no_window, "is_declining_label"].sum() == 0, \
    "pages with no prior window must be unlabelable as declining"
print(f"[OK] all {no_window.sum():,} pages with no prior window are labelled not-declining "
      "by construction -> excluded from the queue rather than ranked low.")

per_client = frame.groupby("client_id")["is_declining_label"].agg(["size", "mean"])
print(f"\nper-client declining rate: min {per_client['mean'].min():.3f}  "
      f"max {per_client['mean'].max():.3f}  (corpus {base:.3f})")
print(f"largest client holds {per_client['size'].max() / len(frame):.1%} of the corpus")
print("-> the reason the queue is worked within a client rather than across clients.")

[OK] no post-refresh outcome column exists -> no recovery claim is possible. (scanned 52 columns)
[OK] 100% of rows are active content -> the model has never seen a dead page.
[OK] all 3,388 pages with no prior window are labelled not-declining by construction -> excluded from the queue rather than ranked low.

per-client declining rate: min 0.000  max 0.937  (corpus 0.542)
largest client holds 23.4% of the corpus
-> the reason the queue is worked within a client rather than across clients.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

**Shipped:** the random forest, out-of-fold scored, top 1,000 of 26,612 eligible pages. The
operating assumption is that a cycle works 500 to 1,000 pages, which is where the forest's advantage
is consistent. If a cycle only ever reaches 100 pages, logistic regression is the better deliverable
and the queue becomes fully explainable.

**The explainability finding.** The hand rule's reason codes explain only 55.6% of what the forest
ranks. The rule fires on thinness, staleness and weak CTR; the forest ranks on sustained visibility
and age, for which the rule has no code. The gap is closed with model-derived codes in a separate
column, so a reviewer always knows which scorer is talking.

**What to do when you get there.** The queue says which page next; it does not say what to do about
it. Each eligible page carries one of eight archetypes drawn from observable page condition, each
pointing at a different kind of edit. Three of them cover 85% of the delivered queue, so a cycle is
three jobs rather than eight, and 285 rows remain `no_clear_defect`, which is the ceiling on how
much of a cycle can be planned in advance. The archetype is triage, not evidence: no refresh outcome
exists in this data.

**The ranking optimises risk, not actionability.** It buries the archetypes it cannot help with
without being told to, and it buries two cheap wins along with them. Risk and actionability are
different quantities and only the first is ranked.

**Read the queue within an age band.** The declining label fires on 0.673 of pages aged 90 to 120
days against 0.439 past 470 days, and it holds inside every impression quartile. That is not a
finding about content: post-launch settling, survivorship, and missing word counts on old pages all
predict the same curve. No refresh cadence is recommended, because `days_since_last_update` has 56
distinct values with 68% of the corpus on two of them. What survives is that 81% of the queue is
content under 180 days old and none of it is older than a year, from 23% of the corpus.

In [6]:
# Section 6 - the delivered queue and what working it is worth.
q = playbook["queue_quality"]
decision = playbook["operating_decision"]
print(f"shipped model : {decision['shipped_model']}, {decision['scoring']}")
print(f"queue depth   : {decision['queue_depth']:,} of {playbook['corpus']['eligible_for_queue']:,} eligible")

print("\n--- what working the queue is worth ---")
print(pd.DataFrame(q).T.to_string())
depth = decision["queue_depth"]
found = q[f"p@{depth}"]["declining_pages_found"]
expected = base * depth
print(f"\nat depth {depth}: {found} declining pages found vs {expected:.0f} expected from an "
      f"arbitrary order")
print(f"  -> {found - expected:.0f} fewer wasted reviews per thousand, "
      f"{q[f'p@{depth}']['lift_vs_base_rate']}x the base rate")

print("\n--- explanation coverage ---")
cov = playbook["explainability"]["coverage_by_depth"]["top_1000"]
print(f"explained by the hand rule's codes : {cov['rule_only_codes']:.1%}")
print(f"explained by something             : {cov['any_code']:.1%}")
print(f"tree rule quoted to reviewers      : "
      f"{playbook['explainability']['depth_2_tree_rule_in_readable_units']}")

print("\n--- action mix across the delivered queue ---")
for action, count in playbook["explainability"]["action_mix"].items():
    print(f"  {action:<30} {count:>4}")

print("\n--- monitoring: the two triggers that matter most ---")
for t in playbook["monitoring_triggers"]["triggers"]:
    if t["name"] in ("queue_precision_decay", "label_definition_change"):
        print(f"\n  {t['name']}")
        print(f"    watch : {t['watch']}")
        print(f"    action: {t['action']}")

shipped model : random_forest, out-of-fold; every page scored by a forest that never saw its client
queue depth   : 1,000 of 26,612 eligible

--- what working the queue is worth ---
        precision  lift_vs_base_rate  declining_pages_found  wasted_reviews
p@100       0.770             1.4205                   77.0            23.0
p@250       0.804             1.4832                  201.0            49.0
p@500       0.782             1.4426                  391.0           109.0
p@1000      0.778             1.4352                  778.0           222.0

at depth 1000: 778 declining pages found vs 542 expected from an arbitrary order
  -> 236 fewer wasted reviews per thousand, 1.4352x the base rate

--- explanation coverage ---
explained by the hand rule's codes : 55.6%
explained by something             : 100.0%
tree rule quoted to reviewers      : days_with_impressions > 12 and content_age_days <= 373 -> at risk

--- action mix across the delivered queue ---
  review_visibility_tre

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed paper is `docs/index.html`: a single self-contained file with its three charts written
as inline SVG, so there are no asset paths to break on GitHub Pages and nothing to load from an
external host.

This section does the check that matters more than regenerating them: it **re-reads the published
page and asserts that every headline number in its prose still matches the receipt it came from.**
A paper that quietly drifts from its own data is the failure mode this whole project has been
arguing against, so it is enforced here rather than trusted.

In [7]:
# Section 7 - verify the deployed paper against the receipts it quotes.
PAPER = ROOT / "docs" / "index.html"
assert PAPER.exists(), "docs/index.html must exist before the paper can be deployed"
html = PAPER.read_text(encoding="utf-8")

arch = playbook["archetypes"]
decay = playbook["content_decay"]
YOUNGEST, OLDEST = list(decay["by_content_age"])[0], list(decay["by_content_age"])[-1]

expected = {
    "0.726": round(grouped["random_forest"]["p@1000"]["mean"], 3),
    "0.776": round(grouped["random_forest"]["p@50"]["mean"], 3),
    "0.558": round(grouped["baseline_rule"]["p@1000"]["mean"], 3),
    "0.542": round(base, 3),
    "0.952": round(naive["random_forest"]["p@50"]["mean"], 3),
    "22.7": audit["split_comparison"]["inflation"]["random_forest"]["p@50"]["relative_inflation_pct"],
    "30,000": recon["rows_reconstructed_exactly"],
    "26,612": playbook["corpus"]["eligible_for_queue"],
    "3,388": playbook["corpus"]["held_out_no_prior_window"],
    "778": q[f"p@{depth}"]["declining_pages_found"],
    "55.6": round(cov["rule_only_codes"] * 100, 1),
    # Added with the archetype and decay recommendations. Only literals distinctive
    # enough not to collide with an SVG coordinate are checked by substring; the
    # archetype table itself is verified row by row below.
    "8,713": arch["order_matters_for_pages"],
    "0.611": round(arch["eligible_base_rate"], 3),
    "0.531": round(arch["buckets"]["stale_authority"]["declining_rate"], 3),
    "0.512": round(arch["buckets"]["weak_engagement"]["declining_rate"], 3),
    "0.673": decay["by_content_age"][YOUNGEST]["declining_rate"],
    "0.439": decay["by_content_age"][OLDEST]["declining_rate"],
    "0.342": decay["by_content_age"][YOUNGEST]["median_prev_30d_share"],
    "0.191": decay["by_content_age"][YOUNGEST]["median_last_30d_share"],
}
def agrees(literal: str, value: float) -> bool:
    """The paper may round; it may not disagree.

    Compare at the precision the paper actually printed, so 22.7 in the prose is
    accepted against a receipt of 22.68 but 22.9 would not be.
    """
    plain = literal.replace(",", "")
    decimals = len(plain.split(".")[1]) if "." in plain else 0
    return round(float(plain), decimals) == round(float(value), decimals)


print("--- every headline figure in the paper, checked against its receipt ---")
problems = []
for literal, value in expected.items():
    in_paper = literal in html
    matches = agrees(literal, value)
    status = "OK " if (in_paper and matches) else "PROBLEM"
    if status == "PROBLEM":
        problems.append((literal, value, in_paper, matches))
    note = "" if str(value) == literal.replace(",", "") else f"  (rounded from {value})"
    print(f"  {status} {literal:<8} in page: {str(in_paper):<5} receipt says: {value}{note}")
assert not problems, f"paper and receipts disagree: {problems}"

# The required structure of a finished paper.
sections = re.findall(r"<h2>(.*?)</h2>", html, re.S)
required = ["abstract", "data", "methodolog", "result", "limitation",
            "recommendation", "reproducib", "acknowledg"]
lowered = " ".join(sections).lower()
missing_sections = [name for name in required if name not in lowered]
assert not missing_sections, f"paper is missing sections: {missing_sections}"
assert 'href="https://flyrank.ai"' in html, "the data credit must link to flyrank.ai"

print(f"\n[OK] {len(sections)} sections present, all required ones found:")
for s in sections:
    print(f"     - {re.sub(r'<[^>]+>', '', s).strip()}")
print("[OK] data credit links to flyrank.ai")
print(f"[OK] page is self-contained: {len(re.findall(r'<svg', html))} inline SVG charts, "
      f"{len(html.encode('utf-8')) / 1024:.0f} KB")

# The archetype table is eight rows of numbers a reader will trust, so it is
# checked against the receipt cell by cell rather than by substring.
table_rows = re.findall(
    r"<tr><td><code>(\w+)</code></td><td>[^<]+</td><td>(\d+)</td><td>([\d.]+)%</td></tr>", html
)
assert len(table_rows) == len(arch["buckets"]), (
    f"paper shows {len(table_rows)} archetype rows, receipt has {len(arch['buckets'])}"
)
print("\n--- the paper's archetype table, checked row by row ---")
for name, rows, share in table_rows:
    receipt = arch["buckets"][name]
    assert int(rows) == receipt["queue_pages"], (
        f"{name}: paper says {rows} rows, receipt says {receipt['queue_pages']}"
    )
    assert round(float(share), 1) == round(receipt["queue_share"] * 100, 1), (
        f"{name}: paper says {share}%, receipt says {receipt['queue_share'] * 100:.1f}%"
    )
    print(f"  OK  {name:<24} {rows:>4} rows  {share:>5}%")
assert sum(int(r) for _, r, _ in table_rows) == playbook["operating_decision"]["queue_depth"], \
    "the archetype rows must account for every delivered page"
print(f"[OK] all {len(table_rows)} archetype rows agree with the receipt and sum to the queue depth")

# The young-content skew is the paper's most actionable claim, so it is asserted.
old_bands = list(decay["queue_age_skew"].values())[-2:]
assert sum(b["queue_share"] for b in old_bands) == 0.0, \
    "the paper claims no page older than 365 days reaches the queue"
print("[OK] the young-content skew claim holds: no page past 365 days is in the queue")

url_file = ROOT / "submission" / "paper_url.txt"
url = url_file.read_text(encoding="utf-8").strip()
print(f"\nsubmission/paper_url.txt -> {url}")
assert url.startswith("https://"), "paper_url.txt must hold one https URL"
assert "PASTE" not in url.upper(), "paper_url.txt still holds the placeholder"
print("[OK] submission URL recorded.")

--- every headline figure in the paper, checked against its receipt ---
  OK  0.726    in page: True  receipt says: 0.726
  OK  0.776    in page: True  receipt says: 0.776
  OK  0.558    in page: True  receipt says: 0.558
  OK  0.542    in page: True  receipt says: 0.542
  OK  0.952    in page: True  receipt says: 0.952
  OK  22.7     in page: True  receipt says: 22.68  (rounded from 22.68)
  OK  30,000   in page: True  receipt says: 30000
  OK  26,612   in page: True  receipt says: 26612
  OK  3,388    in page: True  receipt says: 3388
  OK  778      in page: True  receipt says: 778
  OK  55.6     in page: True  receipt says: 55.6
  OK  8,713    in page: True  receipt says: 8713
  OK  0.611    in page: True  receipt says: 0.611
  OK  0.531    in page: True  receipt says: 0.531
  OK  0.512    in page: True  receipt says: 0.512
  OK  0.673    in page: True  receipt says: 0.6728  (rounded from 0.6728)
  OK  0.439    in page: True  receipt says: 0.439
  OK  0.342    in page: True  receipt

---

# ML-12 — Demo, social cut, and employer summary

*The smallest card and the easiest to forget. Three repackagings of the same work for three
different audiences.*

## A. Five-minute demo outline

| Min | Beat | What is on screen | The line that carries it |
|---|---|---|---|
| **0:00–0:45** | The problem | The queue CSV, scrolling | "30,000 pages, enough review hours for a thousand. Somebody has to pick. Right now that is whoever asked most recently." |
| **0:45–1:30** | The honest baseline | The five hand rules | "I wrote the obvious rule first, on purpose. It finds declining pages 62% of the time in the top 50, against a base rate of 54%. Then it collapses to chance by row 500." |
| **1:30–2:45** | **The leak** (the centrepiece) | The three-bar AUC chart | "Two columns ship in the feature file. Alone, one scores 0.49 and the other 0.62 — nothing. Their ratio scores a perfect 1.000, because the label *is* that ratio. A single-feature scan cannot see a leak that lives in an interaction. I only found it by rebuilding the label from its definition." |
| **2:45–3:45** | The split | The inflation chart | "A random split would have let me claim 0.952. The honest number is 0.776. And look at the ordering: the hand rule *loses* 9.7%, the two-split tree gains 2%, the forest gains 23%. The inflation scales with the model's capacity to memorise a client. That is not a better model, that is a leak." |
| **3:45–4:30** | What it found | The depth-2 tree line | "The rule's heaviest signal, days since last update, ranks 12th to the model. Recency of editing carries almost no information about decline here. The model ranks on sustained visibility and age instead." |
| **4:30–5:00** | The boundary | The limitations section | "It orders review. It does not predict that refreshing recovers traffic, because no refresh outcome exists in this data. I would rather ship that sentence than a bigger number." |

**If there is time for one question:** *"Why not just use logistic regression?"* → "At the top of the
queue I should. It matches the forest at 50 rows. The forest only earns its complexity if you work
the queue hundreds of rows deep, and that is an operating decision, not a modelling one."

---

## B. Social post cut

> Spent 8 weeks ranking 30,000 content pages for refresh review.
>
> The most useful thing I found wasn't the model. It was a leak.
>
> Two columns shipped in the feature file. Alone, they score 0.49 and 0.62 AUC against the label —
> basically noise. Their ratio scores 1.000.
>
> The label *was* that ratio. A single-feature leakage scan can't see it, because the leak lives in
> an interaction. I only caught it by rebuilding the label from its own definition and asking which
> shipped columns appear in it.
>
> Audit your data dictionary, not just your correlation table.
>
> Final honest number: precision@1000 of 0.726 ± 0.054 under a client-grouped split, base rate
> 0.542. A random split would've let me claim 0.952. 🙃
>
> Full write-up and code ↓

*(Pairs with the three-bar AUC chart. One finding, one chart, one method sentence, one link.)*

---

## C. Employer summary, three sentences

> I built an end-to-end content-refresh prioritisation system on 30,000 pages across 32 real
> clients: a transparent hand-rule baseline, three models compared under a client-grouped split, a
> leakage audit, and a delivered 1,000-page review queue with reason codes and monitoring triggers.
> The audit found an exact label reconstruction hiding in two shipped columns that a standard
> single-feature scan could not detect, and showed that a naive row-level split would have
> overstated my headline result by 23%. The shipped model ranks declining pages at precision@1000 of
> 0.726 against a 0.542 base rate, and every number traces to a committed JSON receipt rather than
> to a notebook that has to be re-run to be believed.

---

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and
      **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom. *Asserted in
      section 7.*
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut +
      a 3-sentence employer-facing summary.